# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes
I use the validation period ranking to create a review queue for editors. Pages with higher priority scores are reviewed first. I also simple reason codes so a human can understand whay a page is high in the queue. The reason codes are based on high impressions, low CTR and weaker average search position. These are review signals only and do not mean a page must be changed automatically.

In [2]:
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

apl = HfApi(token=hf_token)

files = apl.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

march_file = [f for f in files if "2026-03" in f][0]

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=march_file,
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(march_path)
df["report_date"] = pd.to_datetime(df["report_date"])

print("Rows loaded:", len(df))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378


In [3]:
# Use the validation period
val_df = df[df["report_date"] > "2026-03-24"].copy()

queue = (
    val_df.groupby("content_hash_id")
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean")
    )
    .reset_index()
)

queue["ctr"] = (
    queue["gsc_clicks"] /
    queue["gsc_impressions"].replace(0,pd.NA)
).fillna(0)

queue["gsc_avg_position"] = queue["gsc_avg_position"].fillna(100)

queue["priority_score"] = (
    queue["gsc_impressions"]*
    (1 - queue["ctr"])*
    queue["gsc_avg_position"]
)

queue = queue.sort_values(
    "priority_score",
    ascending=False
)

print(queue.head(10))

                 content_hash_id  gsc_impressions  gsc_clicks  \
149441  content_73aa61dcedebbf30            27069           2   
70657   content_36e53e9c707674fc            39542          58   
131959  content_66288edeb93b7c4f            79987         422   
131885  content_661a7734f691bef5            39549          11   
169222  content_82e35c4845e6c391            34717          19   
82065   content_3f9e8f387f3fe7e7            19619          12   
324200  content_fa84f5976d5fe3c1            20912          29   
211749  content_a3a1317f7c2bc3dd            21995           1   
221913  content_ab91e088440ace78            18444           0   
25144   content_136c4bf04b07b778            16130          12   

        gsc_avg_position       ctr  priority_score  
149441         49.217629  0.000074    1.332174e+06  
70657          31.773706  0.001467    1.254553e+06  
131959         13.738387  0.005276    1.093095e+06  
131885         26.531121  0.000278    1.048987e+06  
169222         30.1

/tmp/ipykernel_2115/2348345811.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [4]:
queue["reason_code"] = "REVIEW"

queue.loc[
    queue["gsc_impressions"] >= queue["gsc_impressions"].median(),
    "reason_code"
] = "HIGH_IMPRESSIONS"

queue.loc[
    queue["ctr"] <= queue["ctr"].median(),
    "reason_code"
] = queue["reason_code"] + " + LOW_CTR"

queue.loc[
    queue["gsc_avg_position"] >=queue["gsc_avg_position"].median(),
    "reason_code"
] = queue["reason_code"] + " + MAKER_POSITION"

print(
    queue[
        [
            "content_hash_id",
            "priority_score",
            "reason_code"
        ]
    ].head(10)
)

                 content_hash_id  priority_score                 reason_code
149441  content_73aa61dcedebbf30    1.332174e+06            HIGH_IMPRESSIONS
70657   content_36e53e9c707674fc    1.254553e+06            HIGH_IMPRESSIONS
131959  content_66288edeb93b7c4f    1.093095e+06            HIGH_IMPRESSIONS
131885  content_661a7734f691bef5    1.048987e+06            HIGH_IMPRESSIONS
169222  content_82e35c4845e6c391    1.047175e+06            HIGH_IMPRESSIONS
82065   content_3f9e8f387f3fe7e7    9.753288e+05            HIGH_IMPRESSIONS
324200  content_fa84f5976d5fe3c1    8.189318e+05            HIGH_IMPRESSIONS
211749  content_a3a1317f7c2bc3dd    7.945984e+05            HIGH_IMPRESSIONS
221913  content_ab91e088440ace78    7.704454e+05  HIGH_IMPRESSIONS + LOW_CTR
25144   content_136c4bf04b07b778    7.686408e+05            HIGH_IMPRESSIONS


In [8]:
queue["action"] = "Review page"


queue.loc[
   queue["reason_code"].str.contains("WEAKER_POSITION"),
   "action"
] = "Review content and search position"

queue.loc[
    queue["reason_code"].str.contains("HIGH_IMPRESSIONS"),
    "action"
] = "prioritize for review"

queue.loc[
    queue["reason_code"].str.contains("LOW_CTR"),
    "action"
] = "Review title and snippet"

print(
    queue[
        [
        "content_hash_id",
        "priority_score",
        "reason_code",
        "action"
       ]
   ] .head(10)
)

                 content_hash_id  priority_score                 reason_code  \
149441  content_73aa61dcedebbf30    1.332174e+06            HIGH_IMPRESSIONS   
70657   content_36e53e9c707674fc    1.254553e+06            HIGH_IMPRESSIONS   
131959  content_66288edeb93b7c4f    1.093095e+06            HIGH_IMPRESSIONS   
131885  content_661a7734f691bef5    1.048987e+06            HIGH_IMPRESSIONS   
169222  content_82e35c4845e6c391    1.047175e+06            HIGH_IMPRESSIONS   
82065   content_3f9e8f387f3fe7e7    9.753288e+05            HIGH_IMPRESSIONS   
324200  content_fa84f5976d5fe3c1    8.189318e+05            HIGH_IMPRESSIONS   
211749  content_a3a1317f7c2bc3dd    7.945984e+05            HIGH_IMPRESSIONS   
221913  content_ab91e088440ace78    7.704454e+05  HIGH_IMPRESSIONS + LOW_CTR   
25144   content_136c4bf04b07b778    7.686408e+05            HIGH_IMPRESSIONS   

                          action  
149441     prioritize for review  
70657      prioritize for review  
131959     pri

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

###Intended use and limits
The queue is intended to help editors decide which pages should be reviewed first. It is a decision support tool not an automatic content system. A high priority score means a page may be worth reviewing but it doe not prove that the page needs to be changed. The ranking is based on the avaliable search performance data and may become less reliable when performance changes overtime. The validation audit also showed that ranking can change between time periods so editors should check the current page and recent performance before taking action.

In [9]:
print("pages in review queue:", len(queue))
print("This queue is for human review only- no automatic content changes.")

pages in review queue: 331436
This queue is for human review only- no automatic content changes.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and the no go list
Editors should check the page, recent search performance and search intent before making changes. The priority score only helps decide what to review first, The system should not automatically rewrite, publish, delete, redirect or change SEO settings . High priority pages still need human review. For cost and value editors should start with high priority pages where the possible impact is larger.


In [10]:
no_go_actions = [
    "automatic rewrite",
    "automatic publish",
    "automatic delete",
    "automatic redirect",
    "automatic SEO changes"
]

print("Human review required before action.")
print("Actions that should NOT be automated:")
for action in no_go_actions:
  print("-", action)


Human review required before action.
Actions that should NOT be automated:
- automatic rewrite
- automatic publish
- automatic delete
- automatic redirect
- automatic SEO changes


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [] Every section above is filled — markdown thinking AND the code that backs it
- [] The notebook runs top to bottom with no errors (Runtime → Run all)
- [] No client names, URLs, or private queries anywhere
- [] My claims use careful words: observed, measured, directional, decision-support
- [] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.